In [1]:
from pathlib import Path

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tif", ".tiff", ".webp"}

data_folder = Path("crop leaf now1")

for folder in sorted(
    (path for path in data_folder.rglob("*") if path.is_dir()),
    key=lambda path: str(path).lower()
):
    image_count = sum(
        1
        for item in folder.iterdir()
        if item.is_file() and item.suffix.lower() in IMAGE_EXTENSIONS
    )

    relative_path = folder.relative_to(data_folder)
    indentation = "  " * (len(relative_path.parts) - 1)

    print(f"{indentation}{folder.name}/ [{image_count} images]")

_quarantine/ [0 images]
  duplicate_across_splits/ [0 images]
    test/ [0 images]
      Bell_pepper__Bell_pepper_Bacterial_spot/ [270 images]
      Bell_pepper__Bell_pepper_Healthy/ [375 images]
      Cabbage__Alternaria_Leaf_Spot/ [1 images]
      Cabbage__Bacterial_spot_rot/ [3 images]
      Cabbage__Black_Rot/ [2 images]
      Cabbage__Cabbage_aphid_colony/ [1 images]
      Cabbage__club_root/ [2 images]
      Cabbage__Downy_Mildew/ [1 images]
      Cabbage__ring_spot/ [1 images]
      Corn__Corn_Common_rust/ [308 images]
      Corn__Corn_Gray_leaf_spot/ [135 images]
      Corn__Corn_Healthy/ [308 images]
      Corn__Corn_Northern_Leaf_Blight/ [249 images]
      Lettuce__Bacterial/ [40 images]
      Lettuce__fungal/ [47 images]
      Lettuce__healthy/ [61 images]
      Potato__Potato_Early_blight/ [253 images]
      Potato__Potato_Healthy/ [27 images]
      Potato__Potato_Late_blight/ [234 images]
      Tomato__Tomato_Bacterial_spot/ [542 images]
      Tomato__Tomato_Early_blight/ 

## Pool train/val/test for the 24 kept classes and re-split 80/10/10.

Near-duplicate images (for example offline augmented copies of the same leaf)
are clustered with perceptual hashes, and each cluster is kept inside ONE split.
This prevents the leakage that a plain random split would bring back.



In [2]:

from pathlib import Path
import shutil
from multiprocessing.pool import ThreadPool

import imagehash
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold
from tqdm import tqdm
import hashlib
import io
import random
 
 

# ----------- CONFIG ----------
SRC_ROOT = Path("crop leaf now1")  
OUT_ROOT = Path("data_split")  
SEED = 42
HASH_THRESHOLD = 4               # max Hamming distance (out of 64 bits) to call two images near-duplicates
GROUP_AWARE = True               # False = plain stratified random split (leakage possible)
N_WORKERS = 4
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

KEEP = [
    "Bell_pepper__Bell_pepper_Bacterial_spot",
    "Bell_pepper__Bell_pepper_Healthy",
    "Brinjal__Diseased_Brinjal_Leaf_-_Cercospora_Leaf_Spot",
    "Brinjal__Fresh_Brinjal_Leaf",
    "Corn__Corn_Common_rust",
    "Corn__Corn_Gray_leaf_spot",
    "Corn__Corn_Healthy",
    "Corn__Corn_Northern_Leaf_Blight",
    "Lettuce__Bacterial",
    "Lettuce__fungal",
    "Lettuce__healthy",
    "Potato__Potato_Early_blight",
    "Potato__Potato_Healthy",
    "Potato__Potato_Late_blight",
    "Tomato__Tomato_Bacterial_spot",
    "Tomato__Tomato_Early_blight",
    "Tomato__Tomato_Healthy",
    "Tomato__Tomato_Late_blight",
    "Tomato__Tomato_Leaf_Mold",
    "Tomato__Tomato_Mosaic_virus",
    "Tomato__Tomato_Septoria_leaf_spot",
    "Tomato__Tomato_Spider_mites",
    "Tomato__Tomato_Target_Spot",
    "Tomato__Tomato_Yellow_Leaf_Curl_Virus",
]

# The 8 orientations (rotations and flips), so rotated or flipped copies are caught
ORIENTS = [
    None,
    Image.Transpose.ROTATE_90,
    Image.Transpose.ROTATE_180,
    Image.Transpose.ROTATE_270,
    Image.Transpose.FLIP_LEFT_RIGHT,
    Image.Transpose.FLIP_TOP_BOTTOM,
    Image.Transpose.TRANSPOSE,
    Image.Transpose.TRANSVERSE,
]


In [3]:
POPCOUNT = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)


def hash_one(path):
    """Return (path, list of 8 uint64-compatible hashes) or (path, None) if unreadable."""
    try:
        with Image.open(path) as im:
            g = im.convert("L").resize((64, 64), Image.Resampling.LANCZOS)
        hashes = []
        for t in ORIENTS:
            v = g if t is None else g.transpose(t)
            hashes.append(int(str(imagehash.phash(v)), 16))
        return path, hashes
    except Exception:
        return path, None


def popcount64(x):
    """Bit count of a uint64 array, returns same shape."""
    x = np.ascontiguousarray(x)
    b = x.view(np.uint8).reshape(x.shape + (8,))
    return POPCOUNT[b].sum(axis=-1, dtype=np.int32)


def cluster_class(H, threshold):
    """Union-find clustering of one class. H has shape (n, 8): 8 orientation hashes per image."""
    n = len(H)
    parent = list(range(n))

    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a

    base = H[:, 0]
    for i in range(n):
        # distance from every image's base hash to image i in each of its 8 orientations
        d = popcount64(base[:, None] ^ H[i][None, :]).min(axis=1)
        for j in np.nonzero(d <= threshold)[0]:
            ri, rj = find(i), find(int(j))
            if ri != rj:
                parent[rj] = ri
    return np.array([find(i) for i in range(n)])


def collect_files():
    rows = []
    for split in ["train", "val", "test"]:
        for cls in KEEP:
            folder = SRC_ROOT / split / cls
            if not folder.is_dir():
                print(f"WARNING: missing folder {folder}")
                continue
            for p in sorted(folder.iterdir()):
                if p.suffix.lower() in IMG_EXTS:
                    rows.append({"src": str(p), "cls": cls, "orig_split": split})
    return pd.DataFrame(rows)


def main():
    if OUT_ROOT.exists():
        raise SystemExit(f"{OUT_ROOT} already exists. Delete it or change OUT_ROOT.")

    # 1. Collect
    df = collect_files()
    print(f"Found {len(df)} images across {df['cls'].nunique()} classes")
    print(df.groupby(["cls", "orig_split"]).size().unstack(fill_value=0))

    # 2. Hash (needed for grouping and for the leakage report)
    with ThreadPool(N_WORKERS) as pool:
        results = list(tqdm(pool.imap(hash_one, df["src"].tolist(), chunksize=64),
                            total=len(df), desc="Hashing"))
    hash_map = dict(results)
    bad = [p for p, h in hash_map.items() if h is None]
    if bad:
        print(f"Skipping {len(bad)} unreadable images, for example: {bad[:5]}")
    df = df[~df["src"].isin(bad)].reset_index(drop=True)
    H_all = np.array([hash_map[p] for p in df["src"]], dtype=np.uint64)  # (N, 8)

    # 3. Cluster near-duplicates within each class
    df["group"] = -1
    next_id = 0
    for cls in KEEP:
        idx = np.where(df["cls"].values == cls)[0]
        labels = cluster_class(H_all[idx], HASH_THRESHOLD)
        _, inv = np.unique(labels, return_inverse=True)
        df.loc[idx, "group"] = inv + next_id
        next_id += inv.max() + 1
    sizes = df.groupby("group").size()
    print(f"\nClusters: {len(sizes)}  | images in multi-image clusters: {(sizes[sizes > 1]).sum()}")

    # 4. Leakage report on the ORIGINAL splits
    orig_sets = df.groupby("group")["orig_split"].agg(set)
    has_train = df["group"].map(lambda g: "train" in orig_sets[g])
    for s in ["val", "test"]:
        m = df["orig_split"] == s
        pct = 100 * (m & has_train).sum() / m.sum()
        print(f"Original {s}: {pct:.1f}% of images have a near-duplicate in original train")

    # 5. Split 80/10/10 (10 folds: fold 0 = test, fold 1 = val, rest = train)
    y = df["cls"].astype("category").cat.codes.values
    groups = df["group"].values if GROUP_AWARE else np.arange(len(df))
    sgkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=SEED)
    folds = [te for _, te in sgkf.split(df, y, groups)]
    df["split"] = "train"
    df.loc[folds[0], "split"] = "test"
    df.loc[folds[1], "split"] = "val"

    # sanity check: no group spans two splits
    if GROUP_AWARE:
        spans = df.groupby("group")["split"].nunique()
        assert (spans == 1).all(), "A duplicate group crosses splits"

    print("\nNew split sizes:")
    print(df["split"].value_counts().reindex(["train", "val", "test"]))
    print((df["split"].value_counts(normalize=True) * 100).round(1).reindex(["train", "val", "test"]))
    print("\nPer class:")
    print(df.groupby(["cls", "split"]).size().unstack(fill_value=0)[["train", "val", "test"]])

    # 6. Copy files (original split is prefixed to avoid filename collisions)
    dests = []
    for _, r in tqdm(df.iterrows(), total=len(df), desc="Copying"):
        src = Path(r["src"])
        dst_dir = OUT_ROOT / r["split"] / r["cls"]
        dst_dir.mkdir(parents=True, exist_ok=True)
        dst = dst_dir / f"{r['orig_split']}_{src.name}"
        shutil.copy2(src, dst)
        dests.append(str(dst))
    df["dest"] = dests

    # 7. Save manifest for later steps (cleaning, tier 3 comparison against the original test set)
    df.to_csv(OUT_ROOT / "manifest.csv", index=False)
    print(f"\nDone. Output in {OUT_ROOT}/  (manifest.csv holds paths, groups, original and new splits)")


if __name__ == "__main__":
    main()

Found 65691 images across 24 classes
orig_split                                          test  train  val
cls                                                                 
Bell_pepper__Bell_pepper_Bacterial_spot              248   2414  321
Bell_pepper__Bell_pepper_Healthy                     228   2809  295
Brinjal__Diseased_Brinjal_Leaf_-_Cercospora_Lea...   157    727  155
Brinjal__Fresh_Brinjal_Leaf                          178    823  176
Corn__Corn_Common_rust                               245   2577  301
Corn__Corn_Gray_leaf_spot                            260   1840  291
Corn__Corn_Healthy                                   216   2440  286
Corn__Corn_Northern_Leaf_Blight                      284   2482  332
Lettuce__Bacterial                                    78    547   88
Lettuce__fungal                                       56    476   65
Lettuce__healthy                                     112    805  117
Potato__Potato_Early_blight                          278   2472  3

Hashing: 100%|██████████| 65691/65691 [01:01<00:00, 1073.26it/s]



Clusters: 32991  | images in multi-image clusters: 50597
Original val: 55.7% of images have a near-duplicate in original train
Original test: 60.3% of images have a near-duplicate in original train

New split sizes:
split
train    52553
val       6569
test      6569
Name: count, dtype: int64
split
train    80.0
val      10.0
test     10.0
Name: proportion, dtype: float64

Per class:
split                                               train  val  test
cls                                                                 
Bell_pepper__Bell_pepper_Bacterial_spot              2387  298   298
Bell_pepper__Bell_pepper_Healthy                     2665  333   334
Brinjal__Diseased_Brinjal_Leaf_-_Cercospora_Lea...    831  104   104
Brinjal__Fresh_Brinjal_Leaf                           941  118   118
Corn__Corn_Common_rust                               2499  312   312
Corn__Corn_Gray_leaf_spot                            1913  239   239
Corn__Corn_Healthy                                   2354  29

Copying: 100%|██████████| 65691/65691 [00:25<00:00, 2538.99it/s]



Done. Output in data_split/  (manifest.csv holds paths, groups, original and new splits)



## Inspect and clean data_split/.

Checks:
  1. Unreadable or truncated images
  2. Very small images
  3. Image size, color mode and background statistics per class and per crop
     (helps spot dataset source shortcuts, for example one crop always 256x256)
  4. Exact duplicate files filed under different classes (label conflicts)
  5. Exact duplicate files that appear in more than one split (leakage)

Bad files are MOVED to data_split/_quarantine/<reason>/..., never deleted,
so everything can be undone by moving files back.

Run:  python 02_inspect_clean.py
Outputs go to artifacts/inspection/ (csv reports and a contact sheet of samples).

In [1]:
import hashlib
import io
import random
import shutil
from multiprocessing.pool import ThreadPool
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
from tqdm import tqdm

# ==== CONFIG ======
DATA_ROOT = Path("data_split")
OUT_DIR = Path("artifacts") / "inspection"
QUARANTINE = DATA_ROOT / "_quarantine"
SPLITS = ["train", "val", "test"]
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
MIN_SIDE = 64          # images with a shorter side than this are quarantined
DO_CLEAN = True        # False = report only, move nothing
N_WORKERS = 8
SEED = 42
SHEET_PER_CLASS = 8    # sample images per class in the contact sheet
THUMB = 96
MIN_PER_SPLIT_WARN = 50
RING_PX = 3            # thin border ring width in a 64x64 thumbnail, about 18 percent of the image area
CORNER_PX = 8          # four corner patches of this size, about 6 percent of the image area
DEDUP_WITHIN_SPLIT = True  # also quarantine extra exact copies inside the same split, keeping one

In [2]:
def inspect_one(task):
    path, split, cls = task
    rec = {"path": path, "split": split, "cls": cls, "ok": False, "md5": None}
    try:
        data = Path(path).read_bytes()
        rec["md5"] = hashlib.md5(data).hexdigest()
        rec["bytes"] = len(data)
        with Image.open(io.BytesIO(data)) as im:
            rec["mode"] = im.mode
            rec["w"], rec["h"] = im.size
            im.load()  # full decode, catches truncated files
            thumb = np.asarray(
                im.convert("RGB").resize((64, 64), Image.Resampling.BILINEAR), dtype=np.float32
            )
        rec["mean_r"], rec["mean_g"], rec["mean_b"] = thumb.reshape(-1, 3).mean(axis=0)
        r, c = RING_PX, CORNER_PX
        ring = np.concatenate([
            thumb[:r].reshape(-1, 3),
            thumb[-r:].reshape(-1, 3),
            thumb[r:-r, :r].reshape(-1, 3),
            thumb[r:-r, -r:].reshape(-1, 3),
        ])
        corners = np.concatenate([
            thumb[:c, :c].reshape(-1, 3),
            thumb[:c, -c:].reshape(-1, 3),
            thumb[-c:, :c].reshape(-1, 3),
            thumb[-c:, -c:].reshape(-1, 3),
        ])
        rec["border_r"], rec["border_g"], rec["border_b"] = ring.mean(axis=0)
        rec["corner_r"], rec["corner_g"], rec["corner_b"] = corners.mean(axis=0)
        rec["ok"] = True
    except Exception as e:
        rec["error"] = str(e)[:120]
    return rec


def collect_tasks():
    tasks = []
    for split in SPLITS:
        split_dir = DATA_ROOT / split
        if not split_dir.is_dir():
            raise SystemExit(f"Missing folder: {split_dir}")
        for cls_dir in sorted(p for p in split_dir.iterdir() if p.is_dir()):
            for p in sorted(cls_dir.iterdir()):
                if p.suffix.lower() in IMG_EXTS:
                    tasks.append((str(p), split, cls_dir.name))
    return tasks


def dominant(series):
    vc = series.value_counts()
    return vc.index[0], vc.iloc[0] / len(series)


def summarize(df, key):
    rows = []
    for name, g in df.groupby(key):
        size, share = dominant(g["w"].astype(str) + "x" + g["h"].astype(str))
        rows.append({
            key: name,
            "n": len(g),
            "unique_sizes": (g["w"].astype(str) + "x" + g["h"].astype(str)).nunique(),
            "dominant_size": size,
            "dominant_share": round(share, 3),
            "median_kb": round(g["bytes"].median() / 1024, 1),
            "mean_rgb": tuple(int(v) for v in g[["mean_r", "mean_g", "mean_b"]].mean()),
            "border_rgb": tuple(int(v) for v in g[["border_r", "border_g", "border_b"]].mean()),
            "corner_rgb": tuple(int(v) for v in g[["corner_r", "corner_g", "corner_b"]].mean()),
        })
    return pd.DataFrame(rows)


def make_contact_sheet(df, path):
    classes = sorted(df["cls"].unique())
    label_w = 300
    sheet = Image.new("RGB", (label_w + SHEET_PER_CLASS * THUMB, len(classes) * THUMB), "white")
    draw = ImageDraw.Draw(sheet)
    rng = random.Random(SEED)
    for r, cls in enumerate(classes):
        paths = df[(df["cls"] == cls) & (df["split"] == "train")]["path"].tolist()
        rng.shuffle(paths)
        draw.text((6, r * THUMB + THUMB // 2 - 5), cls.replace("__", " / ")[:46], fill="black")
        for c, p in enumerate(paths[:SHEET_PER_CLASS]):
            try:
                with Image.open(p) as im:
                    t = im.convert("RGB").resize((THUMB, THUMB))
                sheet.paste(t, (label_w + c * THUMB, r * THUMB))
            except Exception:
                pass
    sheet.save(path)


def main():
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    tasks = collect_tasks()
    n_cls = len({t[2] for t in tasks})
    print(f"Folder: {DATA_ROOT.resolve()} | {n_cls} classes | scanning {len(tasks)} images")
    with ThreadPool(N_WORKERS) as pool:
        recs = list(tqdm(pool.imap(inspect_one, tasks, chunksize=64), total=len(tasks), desc="Inspecting"))
    df = pd.DataFrame(recs)
    df.to_csv(OUT_DIR / "inspection_report.csv", index=False)
    if not df["ok"].any():
        first = df["error"].dropna().iloc[0] if "error" in df else "unknown"
        raise SystemExit(f"Every image failed to load, so this is a code or environment problem, "
                         f"not bad data. First error: {first}")

    # 1 and 2: unreadable and tiny images
    bad = df[~df["ok"]]
    print(f"\nUnreadable images: {len(bad)}")
    if len(bad):
        print(bad[["path", "error"]].head(10).to_string(index=False))
    good = df[df["ok"]].copy()
    good[["w", "h"]] = good[["w", "h"]].astype(int)
    good["min_side"] = good[["w", "h"]].min(axis=1)
    small = good[good["min_side"] < MIN_SIDE]
    print(f"Images with a side shorter than {MIN_SIDE}px: {len(small)}")
    print("Color modes:", good["mode"].value_counts().to_dict(), "(non RGB files are converted to RGB at load time)")

    # 3: size and background statistics
    cls_summary = summarize(good, "cls")
    cls_summary.to_csv(OUT_DIR / "class_summary.csv", index=False)
    good["crop"] = good["cls"].str.split("__").str[0]
    crop_summary = summarize(good, "crop")
    crop_summary.to_csv(OUT_DIR / "crop_summary.csv", index=False)
    pd.set_option("display.width", 200)
    pd.set_option("display.max_columns", 20)
    pd.set_option("display.max_colwidth", 60)
    print("\nPer crop (dominant_share close to 1.0 means almost every image has the same size):")
    print(crop_summary.to_string(index=False))
    print("\nPer class:")
    print(cls_summary.to_string(index=False))

    # 4: exact duplicates under different labels
    to_move = {}  # path -> reason
    for p in bad["path"]:
        to_move[p] = "corrupt"
    for p in small["path"]:
        to_move.setdefault(p, "too_small")

    g = good[~good["path"].isin(to_move)]
    dup = g[g.duplicated("md5", keep=False)]
    conflict_md5 = dup.groupby("md5")["cls"].nunique()
    conflict_md5 = conflict_md5[conflict_md5 > 1].index
    conflict = dup[dup["md5"].isin(conflict_md5)]
    print(f"\nExact duplicates with conflicting labels: {len(conflict)} files in {len(conflict_md5)} groups")
    for p in conflict["path"]:
        to_move.setdefault(p, "label_conflict")

    # 5: exact duplicates across splits (same class), keep the copy in the highest priority split
    rest = dup[~dup["md5"].isin(conflict_md5)]
    priority = {"train": 0, "val": 1, "test": 2}
    leak_count = 0
    for md5, grp in rest.groupby("md5"):
        if grp["split"].nunique() > 1:
            keep_split = min(grp["split"], key=lambda s: priority[s])
            for p in grp[grp["split"] != keep_split]["path"]:
                to_move.setdefault(p, "duplicate_across_splits")
                leak_count += 1
    print(f"Exact duplicate files leaking across splits: {leak_count}")

    # 5b: extra exact copies that sit inside one split (they count twice in val/test metrics)
    if DEDUP_WITHIN_SPLIT:
        left = rest[~rest["path"].isin(to_move)]
        left = left[left.duplicated("md5", keep=False)]
        within = 0
        for md5, grp in left.groupby("md5"):
            for p in sorted(grp["path"])[1:]:
                to_move.setdefault(p, "duplicate_within_split")
                within += 1
        print(f"Extra exact copies inside the same split: {within}")

    # Clean
    moves = pd.DataFrame({"path": list(to_move.keys()), "reason": list(to_move.values())})
    moves.to_csv(OUT_DIR / "quarantine_list.csv", index=False)
    print("\nFiles flagged for quarantine by reason:")
    print(moves["reason"].value_counts().to_string() if len(moves) else "none")
    if DO_CLEAN and len(moves):
        meta = df.set_index("path")[["split", "cls"]]
        for p, reason in tqdm(to_move.items(), desc="Quarantining"):
            split, cls = meta.loc[p, "split"], meta.loc[p, "cls"]
            dst_dir = QUARANTINE / reason / split / cls
            dst_dir.mkdir(parents=True, exist_ok=True)
            shutil.move(p, str(dst_dir / Path(p).name))
        print(f"Moved {len(moves)} files to {QUARANTINE}")
    elif len(moves):
        print("DO_CLEAN is False, nothing was moved.")

    # Final counts and contact sheet
    remaining = good[~good["path"].isin(to_move)] if DO_CLEAN else good
    counts = remaining.groupby(["cls", "split"]).size().unstack(fill_value=0).reindex(columns=SPLITS, fill_value=0)
    print("\nCounts after cleaning:")
    print(counts.to_string())
    low = counts[(counts["val"] < MIN_PER_SPLIT_WARN) | (counts["test"] < MIN_PER_SPLIT_WARN)]
    if len(low):
        print(f"\nWARNING: val or test below {MIN_PER_SPLIT_WARN} images for:")
        print(low.to_string())
    counts.to_csv(OUT_DIR / "counts_after_cleaning.csv")

    make_contact_sheet(remaining, OUT_DIR / "contact_sheet.png")
    print(f"\nReports and contact_sheet.png saved in {OUT_DIR}/")
    print("Open contact_sheet.png and look for wrong labels, non-leaf photos and odd backgrounds.")


if __name__ == "__main__":
    main()

Folder: /Users/Gabriel_Yashim/Documents/Mr Ebenezer/Crop-Prediction-App/data_split | 24 classes | scanning 65691 images


Inspecting: 100%|██████████| 65691/65691 [00:19<00:00, 3378.41it/s]



Unreadable images: 0
Images with a side shorter than 64px: 0
Color modes: {'RGB': 65691} (non RGB files are converted to RGB at load time)

Per crop (dominant_share close to 1.0 means almost every image has the same size):
       crop     n  unique_sizes dominant_size  dominant_share  median_kb        mean_rgb      border_rgb      corner_rgb
Bell_pepper  6315             1       256x256             1.0       22.6 (123, 132, 109) (155, 146, 148) (156, 146, 150)
    Brinjal  2216             1       256x256             1.0       13.3 (113, 120, 108)    (85, 89, 81)    (63, 66, 60)
       Corn 11554             1       256x256             1.0       17.8  (111, 125, 95)   (97, 108, 87)   (91, 101, 83)
    Lettuce  2344             1       256x256             1.0       18.0  (116, 138, 72)  (110, 126, 76)  (109, 123, 78)
     Potato  8518             1       256x256             1.0       24.2 (126, 130, 110) (147, 139, 144) (143, 136, 142)
     Tomato 34744             1       256x256     

Quarantining: 100%|██████████| 13159/13159 [00:02<00:00, 5252.88it/s]


Moved 13159 files to data_split/_quarantine

Counts after cleaning:
split                                                  train  val  test
cls                                                                    
Bell_pepper__Bell_pepper_Bacterial_spot                 1963  244   245
Bell_pepper__Bell_pepper_Healthy                        2026  256   255
Brinjal__Diseased_Brinjal_Leaf_-_Cercospora_Leaf_Spot    831  104   104
Brinjal__Fresh_Brinjal_Leaf                              941  118   118
Corn__Corn_Common_rust                                  1992  249   249
Corn__Corn_Gray_leaf_spot                               1694  211   211
Corn__Corn_Healthy                                      1860  232   232
Corn__Corn_Northern_Leaf_Blight                         2052  255   256
Lettuce__Bacterial                                       502   62    66
Lettuce__fungal                                          402   46    45
Lettuce__healthy                                         718   93   


### Shortcut probe

**Question:** can the class be predicted from crude statistics that contain no lesion
information at all, such as border color or JPEG file size? If yes, a CNN can use the
same cues, and its test score will overstate how well it recognizes disease.
(Geirhos et al., 2020, "Shortcut learning in deep neural networks", arXiv:2004.07780)

It trains a small gradient boosting model on features from artifacts/inspection/inspection_report.csv
(written by 02_inspect_clean.py), fits on the train split and scores on the VAL split.
The test split is not used.


In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score

# ==== CONFIG =====
REPORT = Path("artifacts") / "inspection" / "inspection_report.csv"
SEED = 0
# Areas refer to the share of the image covered (see RING_PX and CORNER_PX in 02_inspect_clean.py).
# Corners are the purest background cue. A ring covering a large share of the image also contains
# leaf tissue, so it can carry real disease color and is weaker evidence of a shortcut.
FEATURE_SETS = {
    "color + border + corners + file size": ["mean_r", "mean_g", "mean_b", "border_r", "border_g", "border_b",
                                              "corner_r", "corner_g", "corner_b", "bytes"],
    "thin border only (18% of image)": ["border_r", "border_g", "border_b"],
    "corners only (6% of image)": ["corner_r", "corner_g", "corner_b"],
    "file size only": ["bytes"],
}
PER_CLASS_SETS = ["thin border only (18% of image)", "corners only (6% of image)"]
# ====


def main():
    df = pd.read_csv(REPORT)
    if "corner_r" not in df.columns:
        raise SystemExit("The report has no corner features. Re-run 02_inspect_clean.py first (about 15 seconds).")
    df = df[df["ok"]]
    df = df[df["path"].map(lambda p: Path(p).exists())].reset_index(drop=True)  # drop quarantined files
    df["crop"] = df["cls"].str.split("__").str[0]

    classes = sorted(df["cls"].unique())
    cls_idx = {c: i for i, c in enumerate(classes)}
    y = df["cls"].map(cls_idx).values
    crop_of_class = np.array([c.split("__")[0] for c in classes])
    same_crop = crop_of_class[:, None] == crop_of_class[None, :]  # (n_classes, n_classes) bool
    classes_in_crop = same_crop.sum(axis=1)

    tr = (df["split"] == "train").values
    va = (df["split"] == "val").values
    y_va = y[va]
    chance_all = 1 / len(classes)
    chance_within = float(np.mean(1 / classes_in_crop[y_va]))
    print(f"{len(df)} images | {len(classes)} classes | val {va.sum()}")
    print(f"Chance level: {chance_all:.3f} over all classes, {chance_within:.3f} within the true crop\n")

    rows, per_class = [], {}
    for name, cols in FEATURE_SETS.items():
        clf = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1, random_state=SEED)
        clf.fit(df.loc[tr, cols].values, y[tr])
        proba = clf.predict_proba(df.loc[va, cols].values)
        pred_all = proba.argmax(1)
        # within crop: only allow classes of the true crop (the crop itself is easy for a CNN)
        masked = proba * same_crop[y_va]
        pred_within = masked.argmax(1)
        rows.append({
            "features": name,
            "acc_all_classes": round(accuracy_score(y_va, pred_all), 3),
            "macro_f1_all": round(f1_score(y_va, pred_all, average="macro"), 3),
            "acc_within_crop": round(accuracy_score(y_va, pred_within), 3),
            "macro_f1_within_crop": round(f1_score(y_va, pred_within, average="macro"), 3),
        })
        if name in PER_CLASS_SETS:
            per_class[name] = [float((pred_within[y_va == i] == i).mean()) for i in range(len(classes))]

    out = pd.DataFrame(rows)
    pd.set_option("display.width", 200)
    pd.set_option("display.max_colwidth", 60)
    print(out.to_string(index=False))
    out.to_csv(REPORT.parent / "shortcut_probe.csv", index=False)
    table = pd.DataFrame({"cls": classes, "chance_within_crop": 1 / classes_in_crop})
    for name in PER_CLASS_SETS:
        table["recall: " + name.split(" (")[0]] = per_class[name]
    table = table.round(3).sort_values(table.columns[-1], ascending=False)
    print("\nRecall within crop for the background-like features (far above chance_within_crop means separable by that cue):")
    print(table.to_string(index=False))
    table.to_csv(REPORT.parent / "shortcut_probe_per_class.csv", index=False)

    print("""
How to read this:
  * Near chance: these cues carry little class information, which is what you want.
  * Far above chance on 'corners only', 'thin border only' or 'file size only': the classes differ in capture
    conditions or source, not only in disease. Some of this can be legitimate (diseased leaves are
    browner), so treat it as a warning to verify with the contact sheet and Grad-CAM, not as proof.
  * Whatever a model with these crude cues scores is the floor a CNN can reach without looking
    at lesions. Judge the CNN by how far above this floor it goes.""")


if __name__ == "__main__":
    main()

52532 images | 24 classes | val 5259
Chance level: 0.042 over all classes, 0.223 within the true crop

                            features  acc_all_classes  macro_f1_all  acc_within_crop  macro_f1_within_crop
color + border + corners + file size            0.738         0.694            0.825                 0.786
     thin border only (18% of image)            0.514         0.480            0.678                 0.648
          corners only (6% of image)            0.508         0.462            0.677                 0.643
                      file size only            0.165         0.113            0.413                 0.390

Recall within crop for the background-like features (far above chance_within_crop means separable by that cue):
                                                  cls  chance_within_crop  recall: thin border only  recall: corners only
                                   Corn__Corn_Healthy               0.250                     0.957                 0.957
     

## Train several light models, compare them on the VALIDATION set, pick one.

The test set is NOT touched here. It is used once, in the evaluation stage.

Imbalance handling (mild, about 11:1 after the class cut):
  * each epoch uses at most CAP_PER_CLASS random images per class (resampled every epoch)
  * effective-number class weights in the loss (Cui et al., 2019, arXiv:1901.05555)
  * label smoothing (Szegedy et al., 2016, arXiv:1512.00567)
  * model selection by macro F1, not accuracy


In [1]:
import json
import math
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, classification_report, f1_score
from torch.utils.data import DataLoader, Sampler

from common import (ARTIFACTS, AUG_MODE, ImageListDataset, SEED, build_model, cpu_latency_ms, eval_transform,
                    get_device, list_classes, predict_logits, scan_split, seed_everything,
                    train_transform)

# ==== CONFIG =======
MODELS = [                       # timm model names
    "mobilenetv3_small_100",     # about 2.5M parameters, smallest
    "mobilenetv3_large_100",     # about 5M
    "efficientnet_b0",           # about 5M
    "resnet18",                  # about 11M
]
EPOCHS = 12
BATCH_SIZE = 64
LR = 5e-4
WEIGHT_DECAY = 1e-2
LABEL_SMOOTHING = 0.1
WARMUP_EPOCHS = 1
PATIENCE = 100                   # early stopping off by default so every run finishes its cosine schedule;
                                 # set to 4 to stop when val macro F1 has not improved for 4 epochs
CAP_PER_CLASS = 1500             # max images per class per epoch (None = use everything)
BETA = 0.999                     # effective-number beta
NUM_WORKERS = 4                  # set to 0 if you hit multiprocessing errors
PRETRAINED = True                # ImageNet weights, needs internet on first run
SKIP_IF_DONE = True
F1_TOLERANCE = 0.005             # models within this macro F1 of the best are treated as tied (ties go to the smallest file)
# ====

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
class CappedClassSampler(Sampler):
    """Each epoch, draw at most `cap` random images per class. Small classes are used in full."""

    def __init__(self, labels, cap, seed):
        self.by_class = {}
        for i, y in enumerate(labels):
            self.by_class.setdefault(y, []).append(i)
        self.cap = cap if cap is not None else max(len(v) for v in self.by_class.values())
        self.rng = np.random.default_rng(seed)
        self.n = sum(min(len(v), self.cap) for v in self.by_class.values())

    def __len__(self):
        return self.n

    def __iter__(self):
        idx = []
        for v in self.by_class.values():
            v = np.array(v)
            if len(v) > self.cap:
                v = self.rng.choice(v, self.cap, replace=False)
            idx.extend(v.tolist())
        return iter(self.rng.permutation(idx).tolist())


def effective_number_weights(counts, beta):
    counts = np.asarray(counts, dtype=np.float64)
    eff = (1.0 - np.power(beta, counts)) / (1.0 - beta)
    w = 1.0 / eff
    return w / w.sum() * len(counts)  # mean weight of 1


def train_one_model(name, classes, train_ds, val_loader, device, out_dir):
    seed_everything(SEED)
    out_dir.mkdir(parents=True, exist_ok=True)
    num_classes = len(classes)

    sampler = CappedClassSampler(train_ds.labels, CAP_PER_CLASS, SEED)
    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS,
        pin_memory=(device.type == "cuda"), persistent_workers=NUM_WORKERS > 0, drop_last=True,
    )

    # class weights from the per-epoch counts, which is what the model actually sees
    raw = np.bincount(train_ds.labels, minlength=num_classes)
    per_epoch = np.minimum(raw, sampler.cap)
    weights = torch.tensor(effective_number_weights(per_epoch, BETA), dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=LABEL_SMOOTHING)

    model = build_model(name, num_classes, PRETRAINED).to(device)
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    steps_per_epoch = len(train_loader)
    total_steps = EPOCHS * steps_per_epoch
    warmup_steps = max(1, WARMUP_EPOCHS * steps_per_epoch)

    def lr_lambda(step):
        if step < warmup_steps:
            return (step + 1) / warmup_steps
        p = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return max(0.01, 0.5 * (1 + math.cos(math.pi * p)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda") if use_amp else None

    best_f1, best_acc, best_epoch, bad_epochs = -1.0, 0.0, 0, 0
    history = []
    t_start = time.time()
    print(f"\n=== {name} | {n_params:.2f}M params | {len(sampler)} images per epoch | device {device} ===")

    for epoch in range(1, EPOCHS + 1):
        model.train()
        running, seen = 0.0, 0
        for x, y in train_loader:
            x, y = x.to(device, non_blocking=True), y.to(device)
            optimizer.zero_grad(set_to_none=True)
            if use_amp:
                with torch.autocast("cuda", dtype=torch.float16):
                    loss = criterion(model(x), y)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss = criterion(model(x), y)
                loss.backward()
                optimizer.step()
            scheduler.step()
            running += loss.item() * x.size(0)
            seen += x.size(0)

        logits, y_true = predict_logits(model, val_loader, device)
        y_pred = logits.argmax(1).numpy()
        y_true = y_true.numpy()
        f1 = f1_score(y_true, y_pred, average="macro")
        acc = accuracy_score(y_true, y_pred)
        history.append({"epoch": epoch, "train_loss": running / seen, "val_macro_f1": f1, "val_acc": acc})
        print(f"epoch {epoch:2d}/{EPOCHS} | loss {running / seen:.4f} | val macro F1 {f1:.4f} | "
              f"val acc {acc:.4f} | {(time.time() - t_start) / 60:.1f} min")

        if f1 > best_f1:
            best_f1, best_acc, best_epoch, bad_epochs = f1, acc, epoch, 0
            torch.save(model.state_dict(), out_dir / "best.pt")
            best_report = classification_report(y_true, y_pred, target_names=classes,
                                                output_dict=True, zero_division=0)
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print(f"Early stopping, no improvement for {PATIENCE} epochs")
                break

    pd.DataFrame(history).to_csv(out_dir / "history.csv", index=False)
    with open(out_dir / "val_report.json", "w") as f:
        json.dump(best_report, f, indent=2)

    model.load_state_dict(torch.load(out_dir / "best.pt", map_location=device))
    result = {
        "model": name,
        "aug_mode": AUG_MODE,
        "run_dir": out_dir.name,
        "best_val_macro_f1": round(float(best_f1), 5),
        "best_val_acc": round(float(best_acc), 5),
        "best_epoch": best_epoch,
        "epochs_run": len(history),
        "train_minutes": round((time.time() - t_start) / 60, 1),
        "params_millions": round(n_params, 2),
        "size_mb": round((out_dir / "best.pt").stat().st_size / 1e6, 1),
        "cpu_latency_ms": round(cpu_latency_ms(model), 1),
    }
    with open(out_dir / "result.json", "w") as f:
        json.dump(result, f, indent=2)
    return result


def main():
    ARTIFACTS.mkdir(exist_ok=True)
    device = get_device()
    classes = list_classes()
    with open(ARTIFACTS / "class_names.json", "w") as f:
        json.dump(classes, f, indent=2)

    train_items = scan_split("train", classes)
    val_items = scan_split("val", classes)
    print(f"{len(classes)} classes | train {len(train_items)} | val {len(val_items)} | device {device}")

    train_ds = ImageListDataset(train_items, train_transform())
    val_ds = ImageListDataset(val_items, eval_transform())
    val_loader = DataLoader(val_ds, batch_size=128, shuffle=False, num_workers=NUM_WORKERS,
                            pin_memory=(device.type == "cuda"), persistent_workers=NUM_WORKERS > 0)

    results = []
    for name in MODELS:
        out_dir = ARTIFACTS / f"{name}_{AUG_MODE}"
        done = out_dir / "result.json"
        if SKIP_IF_DONE and done.exists():
            print(f"\nSkipping {name}, already trained (delete {done} to retrain)")
            results.append(json.load(open(done)))
            continue
        results.append(train_one_model(name, classes, train_ds, val_loader, device, out_dir))

    df = pd.DataFrame(results).sort_values("best_val_macro_f1", ascending=False)
    df.to_csv(ARTIFACTS / f"comparison_{AUG_MODE}.csv", index=False)
    print(f"\n=== Model comparison (validation set, augmentation: {AUG_MODE}) ===")
    print(df.to_string(index=False))

    # Selection: best val macro F1; among models within F1_TOLERANCE of it, take the smallest file.
    # cpu_latency_ms is PyTorch eager timing on this machine and is only informational. Use the ONNX
    # latency printed by 04_evaluate_export.py to judge deployment speed.
    top = df["best_val_macro_f1"].max()
    tied = df[df["best_val_macro_f1"] >= top - F1_TOLERANCE]
    chosen = tied.sort_values("size_mb").iloc[0]
    reason = (f"highest val macro F1 is {top:.4f}; {len(tied)} model(s) within {F1_TOLERANCE} of it; "
              f"smallest file among them chosen")
    with open(ARTIFACTS / "selection.json", "w") as f:
        json.dump({"best_model": chosen["model"], "run_dir": chosen["run_dir"], "reason": reason}, f, indent=2)
    print(f"\nSelected: {chosen['model']}  ({reason})")
    print("Next: python 04_evaluate_export.py")


if __name__ == "__main__":
    main()

24 classes | train 52553 | val 6569 | device mps

=== mobilenetv3_small_100 | 1.54M params | 32149 images per epoch | device mps ===


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package 

epoch  1/12 | loss 1.3649 | val macro F1 0.8653 | val acc 0.8843 | 1.6 min
epoch  2/12 | loss 0.8640 | val macro F1 0.8970 | val acc 0.9146 | 3.0 min
epoch  3/12 | loss 0.7957 | val macro F1 0.9344 | val acc 0.9429 | 4.4 min
epoch  4/12 | loss 0.7737 | val macro F1 0.9259 | val acc 0.9428 | 5.8 min
epoch  5/12 | loss 0.7490 | val macro F1 0.9267 | val acc 0.9434 | 7.2 min
epoch  6/12 | loss 0.7297 | val macro F1 0.9347 | val acc 0.9513 | 8.7 min
epoch  7/12 | loss 0.7122 | val macro F1 0.9598 | val acc 0.9746 | 10.1 min
epoch  8/12 | loss 0.7002 | val macro F1 0.9604 | val acc 0.9737 | 11.6 min
epoch  9/12 | loss 0.6851 | val macro F1 0.9640 | val acc 0.9790 | 13.0 min
epoch 10/12 | loss 0.6752 | val macro F1 0.9661 | val acc 0.9810 | 14.6 min
epoch 11/12 | loss 0.6713 | val macro F1 0.9693 | val acc 0.9828 | 16.0 min
epoch 12/12 | loss 0.6683 | val macro F1 0.9686 | val acc 0.9822 | 17.4 min

=== mobilenetv3_large_100 | 4.23M params | 32149 images per epoch | device mps ===


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package 

epoch  1/12 | loss 1.4258 | val macro F1 0.9477 | val acc 0.9584 | 3.5 min
epoch  2/12 | loss 0.8036 | val macro F1 0.9637 | val acc 0.9758 | 7.0 min
epoch  3/12 | loss 0.7417 | val macro F1 0.9639 | val acc 0.9773 | 10.0 min
epoch  4/12 | loss 0.7175 | val macro F1 0.9639 | val acc 0.9796 | 13.0 min
epoch  5/12 | loss 0.7056 | val macro F1 0.9570 | val acc 0.9752 | 16.1 min
epoch  6/12 | loss 0.6906 | val macro F1 0.9613 | val acc 0.9776 | 19.3 min
epoch  7/12 | loss 0.6827 | val macro F1 0.9633 | val acc 0.9804 | 22.4 min
epoch  8/12 | loss 0.6745 | val macro F1 0.9667 | val acc 0.9814 | 25.5 min
epoch  9/12 | loss 0.6690 | val macro F1 0.9670 | val acc 0.9826 | 463.2 min
epoch 10/12 | loss 0.6639 | val macro F1 0.9661 | val acc 0.9817 | 466.4 min
epoch 11/12 | loss 0.6613 | val macro F1 0.9680 | val acc 0.9830 | 469.6 min
epoch 12/12 | loss 0.6605 | val macro F1 0.9677 | val acc 0.9831 | 472.8 min

=== efficientnet_b0 | 4.04M params | 32149 images per epoch | device mps ===


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package 

epoch  1/12 | loss 1.4465 | val macro F1 0.9370 | val acc 0.9586 | 6.2 min
epoch  2/12 | loss 0.7906 | val macro F1 0.9588 | val acc 0.9743 | 12.2 min
epoch  3/12 | loss 0.7347 | val macro F1 0.9619 | val acc 0.9752 | 18.2 min
epoch  4/12 | loss 0.7126 | val macro F1 0.9584 | val acc 0.9764 | 24.2 min
epoch  5/12 | loss 0.7020 | val macro F1 0.9575 | val acc 0.9761 | 29.9 min
epoch  6/12 | loss 0.6873 | val macro F1 0.9666 | val acc 0.9799 | 35.6 min
epoch  7/12 | loss 0.6794 | val macro F1 0.9646 | val acc 0.9782 | 41.3 min
epoch  8/12 | loss 0.6721 | val macro F1 0.9694 | val acc 0.9820 | 47.0 min
epoch  9/12 | loss 0.6662 | val macro F1 0.9715 | val acc 0.9842 | 53.0 min
epoch 10/12 | loss 0.6615 | val macro F1 0.9703 | val acc 0.9833 | 59.0 min
epoch 11/12 | loss 0.6591 | val macro F1 0.9695 | val acc 0.9840 | 65.0 min
epoch 12/12 | loss 0.6584 | val macro F1 0.9689 | val acc 0.9831 | 71.0 min

=== resnet18 | 11.19M params | 32149 images per epoch | device mps ===


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package 

epoch  1/12 | loss 1.8672 | val macro F1 0.8756 | val acc 0.9091 | 3.7 min
epoch  2/12 | loss 0.8638 | val macro F1 0.9389 | val acc 0.9578 | 7.1 min
epoch  3/12 | loss 0.7862 | val macro F1 0.9508 | val acc 0.9601 | 10.2 min
epoch  4/12 | loss 0.7552 | val macro F1 0.9568 | val acc 0.9717 | 13.3 min
epoch  5/12 | loss 0.7374 | val macro F1 0.9513 | val acc 0.9688 | 16.5 min
epoch  6/12 | loss 0.7228 | val macro F1 0.9635 | val acc 0.9772 | 19.8 min
epoch  7/12 | loss 0.7096 | val macro F1 0.9611 | val acc 0.9761 | 23.1 min
epoch  8/12 | loss 0.7030 | val macro F1 0.9642 | val acc 0.9778 | 26.4 min
epoch  9/12 | loss 0.6962 | val macro F1 0.9653 | val acc 0.9793 | 29.5 min
epoch 10/12 | loss 0.6908 | val macro F1 0.9655 | val acc 0.9798 | 32.6 min
epoch 11/12 | loss 0.6902 | val macro F1 0.9660 | val acc 0.9802 | 35.7 min
epoch 12/12 | loss 0.6883 | val macro F1 0.9665 | val acc 0.9805 | 38.8 min

=== Model comparison (validation set, augmentation: standard) ===
                model a

### Does the trained CNN rely on the image border instead of the leaf?

Three versions of every VALIDATION image are scored:
  full           the normal image
  border_masked  the outer ring is replaced by the mean color (the model must use the interior)
  border_only    everything except the outer ring is replaced by the mean color

How to read it:
  * full vs border_masked: a small drop means the model does not need the border.
  * border_only: chance is 1 / number of classes. Scores far above chance mean the margin alone
    identifies the class, which a leaf-based disease recognizer should not manage.
  * In tightly framed images the ring still contains leaf tissue, so border_only above chance is a
    warning to investigate, not proof. Compare "robust" and "standard" augmentation runs: the robust
    run should have a lower border_only score if it reduces reliance on the margin.


In [5]:
import json

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score, recall_score
from torch.utils.data import DataLoader

from common import (ARTIFACTS, IMG_SIZE, ImageListDataset, build_model, eval_transform, get_device,
                    list_classes, scan_split)

# ==== CONFIG =====================================
# Run folders to compare. Use the same architecture with different augmentation.
# None means only the run named in artifacts/selection.json.
RUN_DIRS = ["mobilenetv3_large_100_robust", "mobilenetv3_large_100_standard"]
RING_FRACTIONS = [0.02, 0.05, 0.125]   # per side, share of image size
NUM_WORKERS = 4
# ====


def ring_mask(size, frac):
    k = max(1, int(round(size * frac)))
    m = torch.zeros(1, 1, size, size, dtype=torch.bool)
    m[..., :k, :] = True
    m[..., -k:, :] = True
    m[..., :, :k] = True
    m[..., :, -k:] = True
    return m


def ring_area_pct(size, frac):
    k = max(1, int(round(size * frac)))
    return 100 * (1 - ((size - 2 * k) / size) ** 2)


@torch.no_grad()
def collect_predictions(model, loader, device, fractions):
    """One pass over the data, all variants computed on each batch."""
    model.eval()
    masks = {f: ring_mask(IMG_SIZE, f).to(device) for f in fractions}
    names = ["full"] + [f"masked_{f}" for f in fractions] + [f"only_{f}" for f in fractions]
    preds = {n: [] for n in names}
    ys = []
    for x, y in loader:
        x = x.to(device)
        preds["full"].append(model(x).argmax(1).cpu())
        for f, m in masks.items():
            preds[f"masked_{f}"].append(model(x.masked_fill(m, 0.0)).argmax(1).cpu())   # 0 is the mean color
            preds[f"only_{f}"].append(model(x.masked_fill(~m, 0.0)).argmax(1).cpu())
        ys.append(y)
    return {k: torch.cat(v).numpy() for k, v in preds.items()}, torch.cat(ys).numpy()


def per_class_table(y, pred, classes):
    n = len(y)
    n_classes = len(classes)
    recall = recall_score(y, pred, labels=range(n_classes), average=None, zero_division=0)
    true_share = np.bincount(y, minlength=n_classes) / n
    pred_share = np.bincount(pred, minlength=n_classes) / n
    precision = np.divide(recall * true_share, pred_share, out=np.zeros(n_classes), where=pred_share > 0)
    return pd.DataFrame({"cls": classes, "recall": recall, "precision": precision,
                         "true_share": true_share, "pred_share": pred_share}).round(3)


def main():
    classes = list_classes()
    run_dirs = RUN_DIRS or [json.load(open(ARTIFACTS / "selection.json"))["run_dir"]]
    device = get_device()
    val_ds = ImageListDataset(scan_split("val", classes), eval_transform())
    loader = DataLoader(val_ds, batch_size=128, shuffle=False, num_workers=NUM_WORKERS,
                        persistent_workers=NUM_WORKERS > 0)
    print(f"Chance level over {len(classes)} classes: {1 / len(classes):.3f}")
    for f in RING_FRACTIONS:
        print(f"Ring {f}: covers {ring_area_pct(IMG_SIZE, f):.0f}% of the image area")

    out_dir = ARTIFACTS / "reports"
    out_dir.mkdir(exist_ok=True)
    pd.set_option("display.width", 220)
    pd.set_option("display.max_colwidth", 60)
    rows = []
    for run_dir in run_dirs:
        name = json.load(open(ARTIFACTS / run_dir / "result.json"))["model"]
        model = build_model(name, len(classes), pretrained=False)
        model.load_state_dict(torch.load(ARTIFACTS / run_dir / "best.pt", map_location="cpu"))
        model.to(device)
        print(f"\nScoring {run_dir} ...")
        preds, y = collect_predictions(model, loader, device, RING_FRACTIONS)
        full_acc = accuracy_score(y, preds["full"])
        for f in RING_FRACTIONS:
            masked, only = preds[f"masked_{f}"], preds[f"only_{f}"]
            share = np.bincount(only, minlength=len(classes)) / len(only)
            top = int(share.argmax())
            rows.append({
                "run": run_dir,
                "ring": f,
                "area_pct": round(ring_area_pct(IMG_SIZE, f)),
                "full_acc": round(full_acc, 4),
                "masked_acc": round(accuracy_score(y, masked), 4),
                "only_acc": round(accuracy_score(y, only), 4),
                "only_macro_f1": round(f1_score(y, only, average="macro"), 4),
                "sink_class": classes[top].split("__")[-1][:24],
                "sink_share": round(float(share[top]), 3),
            })
            per_class_table(y, only, classes).to_csv(out_dir / f"diag_{run_dir}_ring{f}_border_only.csv", index=False)
            per_class_table(y, masked, classes).to_csv(out_dir / f"diag_{run_dir}_ring{f}_border_masked.csv", index=False)

    summary = pd.DataFrame(rows).sort_values(["ring", "run"])
    summary.to_csv(out_dir / "shortcut_diagnostics_summary.csv", index=False)
    print("\nSummary (compare rows with the same ring width and the same architecture):")
    print(summary.to_string(index=False))
    print(f"\nPer class recall, precision and prediction share are in {out_dir}/diag_*.csv")
    print("""
Reading guide:
  * masked_acc close to full_acc: the model does not need the margin.
  * only_macro_f1 well above chance on the THIN rings: the margin alone carries class information.
    Sink_share near 1 / number of classes or higher than the true share signals a collapsed prediction.
  * If the robust run's thin ring only_macro_f1 is clearly lower than the standard run's, the
    augmentation is reducing reliance on the margin. If they are similar, it is not.""")


if __name__ == "__main__":
    main()

Chance level over 24 classes: 0.042
Ring 0.02: covers 7% of the image area
Ring 0.05: covers 19% of the image area
Ring 0.125: covers 44% of the image area

Scoring mobilenetv3_large_100_robust ...


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package 


Scoring mobilenetv3_large_100_standard ...

Summary (compare rows with the same ring width and the same architecture):
                           run  ring  area_pct  full_acc  masked_acc  only_acc  only_macro_f1     sink_class  sink_share
  mobilenetv3_large_100_robust 0.020         7    0.9783      0.9766    0.1093         0.0604   Corn_Healthy       0.652
mobilenetv3_large_100_standard 0.020         7    0.9810      0.9800    0.0550         0.0159   Corn_Healthy       0.971
  mobilenetv3_large_100_robust 0.050        19    0.9783      0.9766    0.2521         0.2325 Tomato_Healthy       0.506
mobilenetv3_large_100_standard 0.050        19    0.9810      0.9804    0.1280         0.0623 Tomato_Healthy       0.508
  mobilenetv3_large_100_robust 0.125        44    0.9783      0.9673    0.5090         0.5098 Tomato_Healthy       0.210
mobilenetv3_large_100_standard 0.125        44    0.9810      0.9604    0.2862         0.2901 Tomato_Healthy       0.556

Per class recall, precision and 

## Evaluate the selected model on the TEST set and export it for Streamlit.

Produces in artifacts/final/<run folder>/:
  model.onnx            ONNX model for onnxruntime (what the Streamlit app loads)
  model.pt              PyTorch weights (for Grad-CAM, fine-tuning or debugging)
  class_names.json      class index to name mapping
  preprocess.json       exact preprocessing the app must replicate
  metrics.json          headline test metrics with bootstrap confidence intervals
and in artifacts/reports/<run folder>/:
  test_per_class.csv, confusion_matrix.png, top_confusions.csv, confidence_thresholds.csv

Do not tune anything based on these test results. If you change the model after
looking at them, the test set is no longer an honest estimate.

In [6]:
import json
import shutil
import time

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from torch.utils.data import DataLoader

from common import (ARTIFACTS, IMG_SIZE, MEAN, STD, ImageListDataset, build_model, eval_transform,
                    get_device, list_classes, predict_logits, scan_split, seed_everything, SEED)

# ==== CONFIG ======
RUN_DIR_OVERRIDE = "mobilenetv3_large_100_robust"
N_BOOT = 1000            # bootstrap resamples for confidence intervals
NUM_WORKERS = 4
ONNX_OPSET = 17

In [9]:
def bootstrap_macro_f1(y_true, y_pred, rng, n_boot):
    n = len(y_true)
    scores = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        scores.append(f1_score(y_true[idx], y_pred[idx], average="macro"))
    return np.percentile(scores, [2.5, 97.5])


def bootstrap_recall(y_true, y_pred, n_classes, rng, n_boot):
    """95 percent CI of recall for each class, resampling within the class."""
    out = []
    for c in range(n_classes):
        correct = (y_pred[y_true == c] == c).astype(float)
        n = len(correct)
        boots = correct[rng.integers(0, n, (n_boot, n))].mean(axis=1)
        out.append(np.percentile(boots, [2.5, 97.5]))
    return np.array(out)


def plot_confusion(cm, classes, path):
    short = [c.replace("__", " / ")[:34] for c in classes]
    fig, ax = plt.subplots(figsize=(14, 12))
    im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(classes)))
    ax.set_yticks(range(len(classes)))
    ax.set_xticklabels(short, rotation=90, fontsize=7)
    ax.set_yticklabels(short, fontsize=7)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title("Normalized confusion matrix (test set, rows sum to 1)")
    fig.colorbar(im, fraction=0.04)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


def export_onnx(model, path):
    model = model.cpu().eval()
    dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
    kwargs = dict(input_names=["input"], output_names=["logits"],
                  dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
                  opset_version=ONNX_OPSET)
    try:
        torch.onnx.export(model, dummy, str(path), dynamo=False, **kwargs)
    except TypeError:  # older torch without the dynamo argument
        torch.onnx.export(model, dummy, str(path), **kwargs)


def main():
    seed_everything(SEED)
    rng = np.random.default_rng(SEED)

    classes = list_classes()
    n_classes = len(classes)
    if RUN_DIR_OVERRIDE:
        run_dir = RUN_DIR_OVERRIDE
        name = json.load(open(ARTIFACTS / run_dir / "result.json"))["model"]
    else:
        sel = json.load(open(ARTIFACTS / "selection.json"))
        name, run_dir = sel["best_model"], sel["run_dir"]
    print(f"Evaluating {name} (run folder {run_dir}) on the test set")
    final_dir = ARTIFACTS / "final" / run_dir
    report_dir = ARTIFACTS / "reports" / run_dir
    final_dir.mkdir(parents=True, exist_ok=True)
    report_dir.mkdir(parents=True, exist_ok=True)

    device = get_device()
    model = build_model(name, n_classes, pretrained=False)
    model.load_state_dict(torch.load(ARTIFACTS / run_dir / "best.pt", map_location="cpu"))
    model.to(device)

    test_items = scan_split("test", classes)
    test_ds = ImageListDataset(test_items, eval_transform())
    loader = DataLoader(test_ds, batch_size=128, shuffle=False, num_workers=NUM_WORKERS,
                        persistent_workers=NUM_WORKERS > 0)
    logits, y_true = predict_logits(model, loader, device)
    probs = torch.softmax(logits, dim=1).numpy()
    y_true = y_true.numpy()
    y_pred = probs.argmax(1)

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    weighted_f1 = f1_score(y_true, y_pred, average="weighted")
    f1_lo, f1_hi = bootstrap_macro_f1(y_true, y_pred, rng, N_BOOT)
    print(f"\nTest accuracy {acc:.4f} | macro F1 {macro_f1:.4f} (95% CI {f1_lo:.4f} to {f1_hi:.4f}) "
          f"| weighted F1 {weighted_f1:.4f}")

    # Per class report with recall confidence intervals
    rep = classification_report(y_true, y_pred, target_names=classes, output_dict=True, zero_division=0)
    rows = pd.DataFrame({c: rep[c] for c in classes}).T[["precision", "recall", "f1-score", "support"]]
    ci = bootstrap_recall(y_true, y_pred, n_classes, rng, N_BOOT)
    rows["recall_ci_low"], rows["recall_ci_high"] = ci[:, 0], ci[:, 1]
    rows = rows.round(4).sort_values("recall")
    rows.to_csv(report_dir / "test_per_class.csv")
    pd.set_option("display.width", 200)
    pd.set_option("display.max_colwidth", 60)
    print("\nPer class (sorted by recall, weakest first):")
    print(rows.to_string())

    # Confusion matrix and most confused pairs
    cm_raw = confusion_matrix(y_true, y_pred, labels=range(n_classes))
    cm_norm = cm_raw / cm_raw.sum(axis=1, keepdims=True).clip(min=1)
    plot_confusion(cm_norm, classes, report_dir / "confusion_matrix.png")
    off = cm_raw.copy()
    np.fill_diagonal(off, 0)
    pairs = [(classes[i], classes[j], int(off[i, j])) for i in range(n_classes)
             for j in range(n_classes) if off[i, j] > 0]
    pairs = pd.DataFrame(sorted(pairs, key=lambda t: -t[2]), columns=["true", "predicted", "count"])
    pairs.to_csv(report_dir / "top_confusions.csv", index=False)
    print("\nMost frequent confusions:")
    print(pairs.head(10).to_string(index=False))

    # Confidence threshold table, for the "uncertain, please retake the photo" fallback
    conf = probs.max(1)
    thr_rows = []
    for t in [0.0, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
        keep = conf >= t
        thr_rows.append({
            "threshold": t,
            "coverage": round(float(keep.mean()), 4),
            "accuracy_on_accepted": round(float((y_pred[keep] == y_true[keep]).mean()), 4) if keep.any() else None,
        })
    thr = pd.DataFrame(thr_rows)
    thr.to_csv(report_dir / "confidence_thresholds.csv", index=False)
    print("\nConfidence threshold trade-off:")
    print(thr.to_string(index=False))

    # Export
    onnx_path = final_dir / "model.onnx"
    export_onnx(model, onnx_path)
    shutil.copy(ARTIFACTS / run_dir / "best.pt", final_dir / "model.pt")

    # Verify ONNX against PyTorch on one test batch, and time it
    import onnxruntime as ort
    sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    x, _ = next(iter(DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=0)))
    with torch.no_grad():
        ref = model.cpu().eval()(x).numpy()
    out = sess.run(["logits"], {"input": x.numpy()})[0]
    max_diff = float(np.abs(ref - out).max())
    agree = float((ref.argmax(1) == out.argmax(1)).mean())
    one = x[:1].numpy()
    for _ in range(5):
        sess.run(["logits"], {"input": one})
    times = []
    for _ in range(50):
        t0 = time.perf_counter()
        sess.run(["logits"], {"input": one})
        times.append((time.perf_counter() - t0) * 1000)
    onnx_ms = float(np.median(times))
    print(f"\nONNX check: max logit difference {max_diff:.2e}, prediction agreement {agree:.3f}, "
          f"CPU latency {onnx_ms:.1f} ms, size {onnx_path.stat().st_size / 1e6:.1f} MB")
    if agree < 1.0 or max_diff > 1e-3:
        print("WARNING: ONNX output differs from PyTorch more than expected. Investigate before deploying.")

    json.dump(classes, open(final_dir / "class_names.json", "w"), indent=2)
    json.dump({
        "input_name": "input",
        "output_name": "logits",
        "color_mode": "RGB",
        "resize": [IMG_SIZE, IMG_SIZE],
        "resize_note": "resize the whole image to a square, no center crop",
        "scale": "divide pixel values by 255 first",
        "mean": list(MEAN),
        "std": list(STD),
        "layout": "NCHW float32",
        "output": "raw logits, apply softmax",
    }, open(final_dir / "preprocess.json", "w"), indent=2)
    json.dump({
        "model": name,
        "run_dir": run_dir,
        "n_test_images": int(len(y_true)),
        "test_accuracy": round(float(acc), 4),
        "test_macro_f1": round(float(macro_f1), 4),
        "test_macro_f1_ci95": [round(float(f1_lo), 4), round(float(f1_hi), 4)],
        "test_weighted_f1": round(float(weighted_f1), 4),
        "onnx_cpu_latency_ms": round(onnx_ms, 1),
        "onnx_size_mb": round(onnx_path.stat().st_size / 1e6, 1),
        "onnx_max_logit_diff": max_diff,
    }, open(final_dir / "metrics.json", "w"), indent=2)
    print(f"\nSaved deployment files to {final_dir}/ and reports to {report_dir}/")


if __name__ == "__main__":
    main()

Evaluating mobilenetv3_large_100 (run folder mobilenetv3_large_100_robust) on the test set


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package 


Test accuracy 0.9750 | macro F1 0.9668 (95% CI 0.9597 to 0.9728) | weighted F1 0.9749

Per class (sorted by recall, weakest first):
                                                       precision  recall  f1-score  support  recall_ci_low  recall_ci_high
Lettuce__fungal                                           0.9189  0.7556    0.8293     45.0         0.6222          0.8667
Lettuce__Bacterial                                        0.9032  0.8485    0.8750     66.0         0.7576          0.9242
Tomato__Tomato_Bacterial_spot                             0.9581  0.9279    0.9428    222.0         0.8963          0.9595
Corn__Corn_Northern_Leaf_Blight                           0.9455  0.9492    0.9474    256.0         0.9219          0.9766
Potato__Potato_Early_blight                               0.9878  0.9605    0.9739    253.0         0.9368          0.9802
Tomato__Tomato_Late_blight                                0.9528  0.9610    0.9569    231.0         0.9351          0.9827
Lettuc

In [8]:
# install onnx 
!pip install onnx onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 73.2 kB/s  0:02:35m0:00:0100:04
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.5/22.5 MB 98.8 kB/s  0:03:446m0:00:0300:07
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.4/565.4 kB 50.1 kB/s  0:00:09 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 69.5 kB/s  0:01:21 eta 0:00:04m
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [onnx]4/5 [onnx]untime]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ultralytics 8.3.153 requires opencv-python>=4.6.0, which is not installed.
matplotlib 3.7.5 requires numpy<2,>=1.20, but you have numpy 2.5.4 which is incompatible.
scipy 1.15.3 requires numpy<2.5,>=1.23.5, but you have numpy 2.5.4 which is incompa

In [12]:
# show the version of the following libraries

for lib in ["torch", "torchvision", "onnx", "onnxruntime", "timm", "pillow", "imagehash", "tqdm", "matplotlib", "numpy", "pandas", "scikit-learn", "opencv-python"]:
    try:
        mod = __import__(lib)
        print(f"{lib}: {mod.__version__}")
    except ImportError:
        print(f"{lib}: not installed")

torch: 2.7.1
torchvision: 0.22.1
onnx: 1.23.2
onnxruntime: 1.32.0
timm: 1.0.25
pillow: not installed
imagehash: 4.3.2
tqdm: 4.65.0
matplotlib: 3.7.5
numpy: 1.26.4
pandas: 2.2.2
scikit-learn: not installed
opencv-python: not installed
